# **Gold layer Aggregated Tables**

### **Loading of Silver layer tables**

In [0]:
train = spark.table("grocery_catalog.silver.train_clean")
stores = spark.table("grocery_catalog.silver.stores_clean")
transactions = spark.table("grocery_catalog.silver.transactions_clean")
holidays = spark.table("grocery_catalog.silver.holidays_clean")
oil = spark.table("grocery_catalog.silver.oil_clean")

### **Creating Stores dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE `grocery_catalog`.gold.dim_store
USING DELTA AS
SELECT DISTINCT
store_nbr AS store_id,
city,
state,
type,
cluster
FROM `grocery_catalog`.silver.stores_clean;
select * from `grocery_catalog`.gold.dim_store;


store_id,city,state,type,cluster
11,Cayambe,Pichincha,B,6
45,Quito,Pichincha,A,11
31,Babahoyo,Los Rios,B,10
9,Quito,Pichincha,B,6
51,Guayaquil,Guayas,A,17
43,Esmeraldas,Esmeraldas,E,10
7,Quito,Pichincha,D,8
24,Guayaquil,Guayas,D,1
5,Santo Domingo,Santo Domingo De Los Tsachilas,D,4
14,Riobamba,Chimborazo,C,7


### **Creating Product dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dim_product
USING DELTA
AS
SELECT DISTINCT
    id AS product_id,
    family AS product_family
    
FROM `grocery_catalog`.silver.train_clean;
SELECT * FROM `grocery_catalog`.gold.dim_product

product_id,product_family
9,DELI
40,CLEANING
41,DAIRY
100,BABY CARE
109,EGGS
118,LADIESWEAR
124,PERSONAL CARE
131,SEAFOOD
135,BEVERAGES
156,MEATS


### **Creating date dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE `grocery_catalog`.gold.dim_date
USING DELTA AS
SELECT DISTINCT
date,
YEAR(date) AS year,
quarter(date) AS quarter,
MONTH(date) AS month,
WEEKOFYEAR(date) AS week,
DAYOFMONTH(date) AS day
FROM `grocery_catalog`.silver.train_clean;
SELECT * FROM `grocery_catalog`.gold.dim_date

date,year,quarter,month,week,day
1943-01-24,1943,1,1,3,24
1943-03-18,1943,1,3,11,18
1943-03-23,1943,1,3,12,23
1943-04-02,1943,2,4,13,2
1943-04-10,1943,2,4,14,10
1943-06-11,1943,2,6,23,11
1943-06-16,1943,2,6,24,16
1943-07-09,1943,3,7,27,9
1943-08-01,1943,3,8,30,1
1943-08-14,1943,3,8,32,14


### **Creating Promotion dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE `grocery_catalog`.gold.dim_promotion
USING DELTA AS
SELECT DISTINCT
onpromotion
FROM `grocery_catalog`.silver.train_clean;
SELECT * FROM `grocery_catalog`.gold.dim_promotion

onpromotion
32
57
172
157
108
195
183
245
259
672


### **Creating Holiday dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE `grocery_catalog`.gold.dim_holiday
USING DELTA AS
SELECT DISTINCT
date,
type,
locale,
locale_name,
description,
transferred
FROM `grocery_catalog`.silver.holidays_clean;
SELECT * FROM `grocery_catalog`.gold.dim_holiday

date,type,locale,locale_name,description,transferred
1947-10-08,HOLIDAY,NATIONAL,Ecuador,Independencia de Guayaquil,false
1946-04-16,EVENT,NATIONAL,Ecuador,Terremoto Manabi+1,false
1946-08-11,TRANSFER,NATIONAL,Ecuador,Traslado Primer Grito de Independencia,false
1944-06-23,HOLIDAY,LOCAL,Latacunga,Cantonizacion de Latacunga,false
1945-05-08,EVENT,NATIONAL,Ecuador,Dia de la Madre,false
1947-01-01,TRANSFER,NATIONAL,Ecuador,Traslado Primer dia del ano,false
1947-11-06,HOLIDAY,REGIONAL,Santa Elena,Provincializacion Santa Elena,false
1943-12-30,ADDITIONAL,NATIONAL,Ecuador,Primer dia del ano-1,false
1945-12-24,ADDITIONAL,NATIONAL,Ecuador,Navidad+1,false
1944-07-03,EVENT,NATIONAL,Ecuador,Mundial de futbol Brasil: Cuartos de Final,false


### **Creating Oil Price Dimension Table**

In [0]:
%sql
CREATE OR REPLACE TABLE grocery_catalog.gold.dim_oil
USING DELTA
AS
SELECT
    date,
    dcoilwtico AS oil_price
FROM grocery_catalog.silver.oil_clean;
SELECT * FROM `grocery_catalog`.gold.dim_oil

date,oil_price
1942-12-31,null
1943-01-01,93.14
1943-01-02,92.97
1943-01-03,93.12
1943-01-06,93.2
1943-01-07,93.21
1943-01-08,93.08
1943-01-09,93.81
1943-01-10,93.6
1943-01-13,94.27


### **Creating Sales Fact table**

In [0]:
%sql

CREATE OR REPLACE TABLE grocery_catalog.gold.fact_sales
USING DELTA
AS
SELECT
    t.date,
    t.store_nbr,
    t.family,
    t.sales AS sales,
    t.onpromotion,
    COALESCE(tr.transactions,0) AS transactions

FROM grocery_catalog.silver.train_clean t

LEFT JOIN grocery_catalog.silver.transactions_clean tr
ON t.store_nbr = tr.store_nbr
AND t.date = tr.date


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM `grocery_catalog`.gold.fact_sales
WHERE sales > 0
LIMIT 20

date,store_nbr,family,sales,onpromotion,transactions
1942-12-31,25,BEAUTY,2.0,0,770
1942-12-31,25,BEVERAGES,810.0,0,770
1942-12-31,25,BREAD/BAKERY,180.589,0,770
1942-12-31,25,CLEANING,186.0,0,770
1942-12-31,25,DAIRY,143.0,0,770
1942-12-31,25,DELI,71.09,0,770
1942-12-31,25,EGGS,46.0,0,770
1942-12-31,25,FROZEN FOODS,29.654999,0,770
1942-12-31,25,GROCERY I,700.0,0,770
1942-12-31,25,GROCERY II,15.0,0,770


#Daily Sales Metrics
### How much total sales occur each day?
Dashboard Visualization

Line chart → Daily Sales Trend

KPI → Total Sales

In [0]:
df = spark.sql("""

SELECT
d.date,
SUM(f.sales) AS total_sales,
SUM(f.transactions) AS total_transactions,
AVG(f.sales) AS avg_sales_per_item
FROM grocery_catalog.gold.fact_sales f
JOIN grocery_catalog.gold.dim_date d
ON f.date = d.date
GROUP BY d.date
ORDER BY d.date

""")

df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("grocery_catalog.gold.sales_daily_metrics")

In [0]:
display(df)

date,total_sales,total_transactions,avg_sales_per_item
1942-12-31,2511.6189990000003,25410,1.4094382710437712
1943-01-01,496092.41794400004,3076095,278.3908069270483
1943-01-02,361461.2311239999,2590632,202.84019703928166
1943-01-03,354459.67709269986,2590302,198.91115437300778
1943-01-04,477350.1212289998,3087909,267.8732442362513
1943-01-05,519695.4010879999,2985312,291.6360275465768
1943-01-06,336122.8010659999,2494701,188.62110048597077
1943-01-07,318347.77798099996,2386725,178.64634005667787
1943-01-08,302530.80901799985,2375043,169.77037543097634
1943-01-09,258982.00304899988,2190639,145.33221270987647


Databricks visualization. Run in Databricks to view.

#Store Performance Metrics
Dashboard

Top performing stores

Sales by location

In [0]:
df = spark.sql("""

SELECT
s.store_id,
s.city,
s.state,
SUM(f.sales) AS total_sales,
SUM(f.transactions) AS total_transactions,
AVG(f.sales) AS avg_sales
FROM grocery_catalog.gold.fact_sales f
JOIN grocery_catalog.gold.dim_store s
ON f.store_nbr = s.store_id
GROUP BY s.store_id, s.city, s.state
ORDER BY total_sales DESC

""")

df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("grocery_catalog.gold.sales_store_metrics")


In [0]:
df = spark.table("grocery_catalog.gold.sales_store_metrics")
display(df)

store_id,city,state,total_sales,total_transactions,avg_sales
44,Quito,Pichincha,6.2087553250088826E7,240012069,1117.2452539064425
45,Quito,Pichincha,5.449801041667408E7,204636795,980.6739080233585
47,Quito,Pichincha,5.0948310061005935E7,215681730,916.7982088282937
3,Quito,Pichincha,5.048191018508199E7,177089550,908.4054953048656
49,Quito,Pichincha,4.342009578376806E7,150945399,781.3304502945379
46,Quito,Pichincha,4.1896062121634215E7,197673729,753.9059620246566
48,Quito,Pichincha,3.593313027393633E7,168556905,646.6049498656937
51,Guayaquil,Guayas,3.291148953760406E7,94829262,592.2315111495728
8,Quito,Pichincha,3.0494286928015053E7,153053043,548.7347392214614
50,Ambato,Tungurahua,2.865302062469518E7,144686652,515.6017531255881


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#Product Family Sales
Dashboard

Top selling product families

Category demand

In [0]:
df = spark.sql("""

SELECT
p.product_family,
SUM(f.sales) AS total_sales,
COUNT(DISTINCT f.store_nbr) AS stores_selling
FROM grocery_catalog.gold.fact_sales f
JOIN grocery_catalog.gold.dim_product p
ON f.store_nbr = p.product_id
GROUP BY p.product_family
ORDER BY total_sales DESC

""")

df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("grocery_catalog.gold.sales_family_metrics")

#Promotion Effectiveness
Dashboard

Sales with promotion vs without promotion

In [0]:
df = spark.sql("""

SELECT
pr.onpromotion,
SUM(f.unit_sales) AS total_sales,
AVG(f.unit_sales) AS avg_sales
FROM grocery_catalog.gold.fact_sales f
JOIN grocery_catalog.gold.dim_promotion pr
ON f.promotion_key = pr.promotion_key
GROUP BY pr.onpromotion

""")

df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("grocery_catalog.gold.promotion_effectiveness")

#Holiday Impact Analysis
Dashboard

Holiday sales spikes

In [0]:
df = spark.sql("""

SELECT
h.type,
h.locale,
SUM(f.unit_sales) AS total_sales
FROM grocery_catalog.gold.fact_sales f
JOIN grocery_catalog.gold.dim_holiday h
ON f.holiday_key = h.holiday_key
GROUP BY h.type, h.locale
ORDER BY total_sales DESC

""")

df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("grocery_catalog.gold.holiday_sales_impact")

#Oil Price Impact on Sales

Important for economic analysis.

Dashboard

Oil price vs grocery sales

In [0]:
df = spark.sql("""

SELECT
o.date,
o.oil_price,
SUM(f.unit_sales) AS total_sales
FROM grocery_catalog.gold.fact_sales f
JOIN grocery_catalog.gold.dim_oil_price o
ON f.oil_key = o.oil_key
GROUP BY o.date, o.oil_price
ORDER BY o.date

""")

df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("grocery_catalog.gold.oil_price_sales_correlation")

#Weekly Forecast Features


In [0]:
df = spark.sql("""

SELECT
d.year,
d.week_of_year,
SUM(f.unit_sales) AS weekly_sales,
AVG(f.transactions) AS avg_transactions
FROM grocery_catalog.gold.fact_sales f
JOIN grocery_catalog.gold.dim_date d
ON f.date_key = d.date_key
GROUP BY d.year, d.week_of_year
ORDER BY d.year, d.week_of_year

""")

df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("grocery_catalog.gold.sales_forecast_features")

Final Dashboard Layout
Page 1 — Sales Overview

KPI → Total Sales

KPI → Total Transactions

Line chart → Daily Sales Trend

Page 2 — Store Analysis

Bar chart → Top Stores

Map → Sales by State

Page 3 — Product Analysis

Pie chart → Product Family Sales

Bar chart → Category Ranking

Page 4 — Promotions

Sales with promotion vs without promotion

Page 5 — External Factors

Holiday sales impact

Oil price correlation